# Frame Extraction for Bowling Score Detection Dataset

**Purpose.** Take the raw `.MOV` videos uploaded as a Kaggle dataset, sample frames at a fixed interval, organise them by clip type (`ball` vs `car`), and produce a single zip file ready to upload to Roboflow for labelling.

**Inputs.** `/kaggle/input/computer-vision/Vidoes/` containing `main_ball_*.MOV` and `bonus_car_*.MOV` files.

**Outputs.** `/kaggle/working/frames_for_roboflow.zip` — extracted frames in two subfolders: `ball_clips/` and `car_clips/`. Filenames encode the source clip and frame index so labelling stays organised.

**Sampling rate.** Default is one frame every 12 frames at 30 fps source, i.e. ~2.5 fps. With 21 ball clips and 21 car clips averaging ~25 seconds each, this produces roughly 1300-1600 frames total. Adjust `FRAME_STRIDE` below if you want fewer or more.

**Important note on Roboflow free-tier limits.** The free tier currently allows ~500 source images per project at any one time on the public workspace. If you want to label more, either upgrade or trim the sampling. Recommended: start with `FRAME_STRIDE = 20` (about 800 frames total) and only sample more densely if validation accuracy is poor. You can always re-extract with a finer stride later.

## 1. Setup and configuration

In [ ]:
import os
import glob
import shutil
import zipfile
from pathlib import Path

import cv2
from tqdm.auto import tqdm

# ----- CONFIG -----
INPUT_DIR = Path('/kaggle/input/computer-vision/Vidoes')
OUTPUT_DIR = Path('/kaggle/working/frames')
ZIP_PATH = Path('/kaggle/working/frames_for_roboflow.zip')

# Sampling: extract one frame every N source frames.
# At 30 fps source: stride 20 = 1.5 fps, stride 12 = 2.5 fps, stride 6 = 5 fps.
# Start with 20 and increase density only if needed.
FRAME_STRIDE = 20

# Resize long edge to this many pixels. Roboflow re-resizes anyway during export,
# but smaller files = faster upload. 1280 keeps quality fine.
RESIZE_LONG_EDGE = 1280

# JPEG quality: 90 is a good balance.
JPEG_QUALITY = 90

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'ball_clips').mkdir(exist_ok=True)
(OUTPUT_DIR / 'car_clips').mkdir(exist_ok=True)

print(f'Input dir: {INPUT_DIR}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'Frame stride: 1 in every {FRAME_STRIDE} frames')
print(f'Resize long edge to: {RESIZE_LONG_EDGE} px')

## 2. Find all videos

Glob both clip types. Globbing handles missing numbers (you mentioned `bonus_car_15.MOV` is absent and the script must not break on that).

In [ ]:
ball_videos = sorted(glob.glob(str(INPUT_DIR / 'main_ball_*.MOV')))
car_videos = sorted(glob.glob(str(INPUT_DIR / 'bonus_car_*.MOV')))

print(f'Found {len(ball_videos)} ball clips')
for v in ball_videos:
    print(f'  {os.path.basename(v)}')

print(f'\nFound {len(car_videos)} car clips')
for v in car_videos:
    print(f'  {os.path.basename(v)}')

if not ball_videos and not car_videos:
    raise RuntimeError(
        f'No videos found at {INPUT_DIR}. Check that the Kaggle dataset is attached '
        f'and the path matches your dataset structure.'
    )

## 3. Frame extraction function

One function to handle both clip types. The output filename pattern is:

`{clip_type}_{clip_number}_f{frame_index:05d}.jpg`

For example: `ball_03_f00120.jpg` is frame 120 of `main_ball_3.MOV`. This naming makes it easy to filter by clip type in Roboflow and to trace any labeled frame back to its source video.

In [ ]:
def resize_keep_aspect(img, long_edge):
    """Resize so the longer edge equals long_edge, preserving aspect ratio."""
    h, w = img.shape[:2]
    if max(h, w) <= long_edge:
        return img
    scale = long_edge / max(h, w)
    new_w, new_h = int(round(w * scale)), int(round(h * scale))
    return cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)


def extract_frames_from_video(video_path, output_subdir, clip_type, frame_stride):
    """Extract every Nth frame from a video. Returns the number of frames written."""
    # Parse clip number from filename, e.g. 'main_ball_3.MOV' -> '3'
    stem = Path(video_path).stem
    clip_num = stem.split('_')[-1].zfill(2)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f'  WARNING: could not open {video_path}')
        return 0

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    written = 0
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % frame_stride == 0:
            frame = resize_keep_aspect(frame, RESIZE_LONG_EDGE)
            out_name = f'{clip_type}_{clip_num}_f{frame_idx:05d}.jpg'
            out_path = output_subdir / out_name
            cv2.imwrite(
                str(out_path),
                frame,
                [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY]
            )
            written += 1

        frame_idx += 1

    cap.release()
    return written, total_frames, fps

## 4. Run extraction on all videos

In [ ]:
total_written = 0
summary = []

print('=== Ball clips ===')
for v in tqdm(ball_videos):
    written, total, fps = extract_frames_from_video(
        v, OUTPUT_DIR / 'ball_clips', 'ball', FRAME_STRIDE
    )
    total_written += written
    summary.append((os.path.basename(v), total, fps, written))
    print(f'  {os.path.basename(v)}: {total} frames @ {fps:.1f} fps -> {written} extracted')

print('\n=== Car clips ===')
for v in tqdm(car_videos):
    written, total, fps = extract_frames_from_video(
        v, OUTPUT_DIR / 'car_clips', 'car', FRAME_STRIDE
    )
    total_written += written
    summary.append((os.path.basename(v), total, fps, written))
    print(f'  {os.path.basename(v)}: {total} frames @ {fps:.1f} fps -> {written} extracted')

print(f'\n=== TOTAL: {total_written} frames extracted ===')

## 5. Sanity check: spot-look at random frames

Display 4 random ball frames and 4 random car frames to confirm extraction worked and looks sensible. If frames look corrupted, blank, or the wrong content, fix the issue before uploading to Roboflow.

In [ ]:
import random
import matplotlib.pyplot as plt

ball_samples = sorted((OUTPUT_DIR / 'ball_clips').glob('*.jpg'))
car_samples = sorted((OUTPUT_DIR / 'car_clips').glob('*.jpg'))

random.seed(42)
ball_show = random.sample(ball_samples, min(4, len(ball_samples)))
car_show = random.sample(car_samples, min(4, len(car_samples)))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, p in enumerate(ball_show):
    img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    axes[0, i].imshow(img)
    axes[0, i].set_title(p.name, fontsize=9)
    axes[0, i].axis('off')
for i, p in enumerate(car_show):
    img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    axes[1, i].imshow(img)
    axes[1, i].set_title(p.name, fontsize=9)
    axes[1, i].axis('off')

plt.suptitle('Sample extracted frames (top: ball clips, bottom: car clips)', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Zip everything for download

Roboflow accepts a single zip upload. The zip preserves the `ball_clips/` and `car_clips/` folder structure so you can use Roboflow's batch tagging to apply 'ball-clip' and 'car-clip' tags during upload — this is essential for the labelling protocol (yellow pins are labelled in car clips but NOT in ball clips).

In [ ]:
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for img_path in tqdm(sorted(OUTPUT_DIR.rglob('*.jpg'))):
        # Preserve relative folder structure inside the zip
        arcname = img_path.relative_to(OUTPUT_DIR)
        zf.write(img_path, arcname)

size_mb = ZIP_PATH.stat().st_size / (1024 * 1024)
print(f'\nZip created: {ZIP_PATH}')
print(f'Size: {size_mb:.1f} MB')
print(f'Total frames inside: {sum(1 for _ in OUTPUT_DIR.rglob("*.jpg"))}')

## 7. Final summary

In [ ]:
import pandas as pd
df = pd.DataFrame(summary, columns=['video', 'total_frames', 'fps', 'extracted'])
df.loc[len(df)] = ['TOTAL', df['total_frames'].sum(), '-', df['extracted'].sum()]
print(df.to_string(index=False))

## What to do next

1. Click the 'Output' tab on the right side of Kaggle. Find `frames_for_roboflow.zip`. Download it to your laptop.
2. Go to https://roboflow.com and create a free account.
3. Create a new project. Type: 'Object Detection'. Annotation group: 'pin-ball-car'. Add the three classes: `pin`, `ball`, `car`.
4. Upload the zip. When prompted, choose to keep the folder structure as 'tags' — this gives every image a `ball-clips` or `car-clips` tag automatically.
5. Label according to the protocol (see the labelling guide message in chat).
6. When done, generate a dataset version with these settings: image size 640x640, augmentations OFF (Ultralytics applies its own), train/val split 80/20.
7. Export as 'YOLOv8' format. Download the zip.
8. Upload that zip as a new Kaggle Dataset (the one you'll feed to the training notebook in the next message).

Total expected time: 4-8 hours of focused labelling work for ~800 frames. Two team members in parallel cuts this in half.